# pipeline — run a dataset version

The batch entry points are the scripts in `scripts/` (the three workflows run them); this notebook is
the operator's view of one dataset version, in five sections:

1. **State** — what exists on Azure and locally for `config.version`.
2. **Reset** — how to delete a version's products for a completely fresh run (dry run until confirmed).
3. **Work list and dispatch** — the tile list, the static inputs, the `gh workflow run` commands.
4. **One tile end to end** — the ancillary window on the dataset grid (wave 2), the UTM tabulation and
   its partial sums (wave 3), a single-tile reduce, and the tile-for-tile comparison with the v9 pixel table.
5. **Local stages** — ERA5 zonal means, the reduce, the metrics table.

Every deletion is a dry run until you flip `confirm=True` / uncomment the cell; the workflows never
delete anything unless their `start_fresh` box is ticked. Mechanics (ledgers, what a partials row is,
sizing): `pipeline/README.md`.

```text
 1 Get ERA5-Land data        era5_land icechunk repo: one commit per water year (ERA5-Land monthly, EE via xee)
                             + its anomaly group (store - median over all years), one commit
 2 Get ancillary data        ancillary icechunk repo on the DATASET grid: 8 layers, one commit per tile
 3 Process tiles to parquets store window + ancillary window -> 80 m UTM -> pixel table -> PARTIAL SUMS parquet per tile
            |                                        |
            v                                        v
 local:  reduce_partials.py  <----  era5_zonal.py (per range / basin zonal means of the anomaly group)
            |
            v
         the cubes  ->  range_metrics.py  ->  the notebooks under analyses/
```

In [ ]:
import subprocess
import sys

import numpy as np
import pandas as pd
import xarray as xr

from gsro_analysis import aggregate, datacube, era5, paths, settings

config = settings.load_config()   # the dataset version lives in settings.CONFIG_FILE
fs = config.azure_blob_fs
VERSION = config.version
WORK_LIST = paths.PIPELINE / 'tile_data' / f'ancillary_tiles_{VERSION}.txt'
print(VERSION, '| water years', list(config.water_years)[0], '-', list(config.water_years)[-1],
      '| basins stored at HydroBASINS level', settings.BASIN_ATLAS_STORED_LEVEL)

## 1. State — what exists now

In [ ]:
# Azure products of this version (fleet + ERA5 + mirrored cubes) — a listing, nothing is touched
for name, (prefix, n) in datacube.version_products(config).items():
    print(f"{name:18s} {n:7d} objects   {prefix}")

In [ ]:
# the two tile ledgers vs the work list: ancillary commits (wave 2) and partials blobs (wave 3)
tiles = [tuple(map(int, l.split())) for l in open(WORK_LIST) if l.strip() and not l.startswith('#')] if WORK_LIST.exists() else []
done_anc, done_part = datacube.completed_ancillary_tiles(config), datacube.completed_partials_tiles(config)
print(f"work list {len(tiles)} tiles | ancillary committed {len(done_anc)} | partials {len(done_part)} | "
      f"wave 2 remaining {sum(1 for t in tiles if t not in done_anc)} | wave 3 remaining {sum(1 for t in tiles if t in done_anc and t not in done_part)}")

In [ ]:
# ERA5 branch: the icechunk repo's commit ledger — which water years are committed, is the anomaly current
if era5.repo_exists(config):
    st = era5.status(config)
    print(f"committed {sorted(st['years'])} | remaining {st['remaining']} | anomaly: "
          f"{'none' if st['anomaly'] is None else ('STALE' if st['anomaly_stale'] else 'current')}")
else:
    print('no ERA5-Land repository yet:', era5.repo_prefix(config), '(the Get ERA5-Land data workflow creates it)')

In [ ]:
# local products of this version (all regenerable by the stage 2/3 scripts + notebooks)
local = {
    'cubes':            sorted((paths.AGGREGATED / VERSION).glob('*/all_*.nc')),
    'era5 zonal':       sorted((paths.AGGREGATED / VERSION / 'era5_zonal').glob('*.nc')),
    'partials cache':   sorted((paths.AGGREGATED / VERSION / 'partials').glob('*.parquet')),
    'figures':          sorted(paths.OUTPUT_ROOT.glob(f'*/figures/{VERSION}')) + sorted(paths.OUTPUT_ROOT.glob(f'*/*/figures/{VERSION}')),
    'results':          sorted(paths.OUTPUT_ROOT.glob(f'*/results/{VERSION}')) + sorted(paths.OUTPUT_ROOT.glob(f'*/*/results/{VERSION}')),
}
for k, v in local.items():
    print(f"{k:16s} {len(v):5d}  {v[0] if v else ''}{' ...' if len(v) > 1 else ''}")

## 2. Reset — start the version completely fresh

| product | Azure prefix | cost to rebuild |
| --- | --- | --- |
| `partials` | `snowmelt_runoff_onset_analysis/partials/<v>/` | wave 3, ~1 min/tile (no Earth Engine) |
| `pixel_tables` | `snowmelt_runoff_onset_analysis/parquets/<v>/` | nothing (opt-in product) |
| `ancillary_grid` | `snowmelt_runoff_onset_analysis/ancillary/<v>_grid/` (the icechunk repo) | wave 2, ~2.5 min/tile (Earth Engine) |
| `aggregated_mirror` | `snowmelt_runoff_onset_analysis/aggregated/<v>/` | `reduce_partials.py --mirror`, minutes |
| `era5_land` | `snowmelt_runoff_onset_analysis/era5_land/<v>/` (the icechunk repo) | wave 1: one ~1-min job per water year in parallel + a ~20-min anomaly job, then `era5_zonal.py --overwrite` |

The workflows' `start_fresh` boxes do the same resets per wave. Never touched by any of this: the
icechunk dataset, the pyramid, the production inputs. A change of a unit definition alone (a HydroBASINS
level, a new inventory) does not need a reset: `datacube.refresh_unit_layers` recommits the id layers of a
stored tile and the tile is then re-mapped.

In [ ]:
# DRY RUN — the full reset including the ERA5 branch (nothing happens without confirm=True)
FULL_RESET = ('partials', 'pixel_tables', 'ancillary_grid', 'aggregated_mirror', 'era5_land')
plan = datacube.reset_version(config, what=FULL_RESET, confirm=False)

In [ ]:
# ---- THE RESET. Uncomment deliberately. Drop 'era5_land' from the tuple to keep the ERA5 branch. ----
# datacube.reset_version(config, what=FULL_RESET, confirm=True)
# datacube.version_products(config)   # every count should now be 0

In [ ]:
# ---- local products: uncomment to clear (they are regenerated by the stage 2/3 scripts and the notebooks) ----
import shutil
# for d in [paths.AGGREGATED / VERSION] + local['figures'] + local['results']:
#     shutil.rmtree(d, ignore_errors=True)
# print('local products cleared')

## 3. Work list and dispatch

The work list is every tile whose composites hold data, from the icechunk commit history. Regenerate
it whenever the dataset gains tiles (a new water year, a reprocessing); the dispatcher
(`scripts/get_remaining_work.py`) reads this file and diffs it against the two Azure ledgers.

In [ ]:
from global_snowmelt_runoff_onset import status
tile_status_gdf = status.get_tile_status_gdf(config)
tiles_with_data = tile_status_gdf[tile_status_gdf['composites'] == 'data']
print(f"{len(tiles_with_data)} tiles with data (current list: {len(tiles)})")
tiles_with_data.head()

In [ ]:
# ---- (re)write the work list ----
# with open(WORK_LIST, 'w') as f:
#     for _, t in tiles_with_data.iterrows():
#         f.write(f"{int(t['row'])} {int(t['col'])}\n")
# print('wrote', WORK_LIST)

### Static inputs

Downloaded once into `data/geometries/sources/` (the fleet jobs restore them from the Actions cache)
and the GTOPO30 land histogram under `data/` (tracked; grid/version independent; rebuild via Earth
Engine only if the latitude/elevation bins change).

In [ ]:
print(settings.cached_source(settings.GMBA_URL))
print(settings.cached_source(settings.CONTINENTS_URL))
print(settings.basin_atlas_gdb(), '| layer stored by the fleet:', settings.BASIN_ATLAS_LAYER)
gtopo = paths.gtopo30_histogram()
print(gtopo, 'exists' if gtopo.exists() else 'MISSING -> run reduce_partials.py --build-gtopo30 (Earth Engine)')

### Dispatch the three workflows

All are `workflow_dispatch` workflows of this repository (`.github/workflows/`), run on GitHub-hosted
runners with the repository secrets `AZURE_STORAGE_SAS_TOKEN`, `AZURE_STORAGE_ACCOUNT` and
`EE_SERVICE_ACCOUNT_KEY`. Dispatch them in order, each until its plan job reports 0 remaining: every
run is idempotent (a done tile or year is skipped, a failed one is re-listed).

In [ ]:
REPO = 'egagli/global_snowmelt_runoff_onset_analysis'
def gh(*args):
    """Run a gh CLI command and show BOTH streams — a failed dispatch (auth, bad input name,
    workflow not on the default branch) otherwise looks like nothing happened."""
    r = subprocess.run(['gh', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip() or f'(no output, exit {r.returncode})')
    return r.returncode
gh('auth', 'status')
gh('run', 'list', '-R', REPO, '--limit', '8')

In [ ]:
# ---- wave 1: Get ERA5-Land data (idempotent: creates the repo if missing, fetches only the water years without a
#      commit, then builds the anomaly group once all are committed). start_fresh=true DELETES the version's repo first.
# gh('workflow', 'run', 'get_era5_land_data.yml', '-R', REPO, '-f', 'start_fresh=false', '-f', 'rebuild_anomaly=false')

In [ ]:
# ---- wave 2: Get ancillary data (batch_size 36 -> ~120 jobs of ~1.5 h; start_fresh=true DELETES the ancillary repo first)
# gh('workflow', 'run', 'get_ancillary_data.yml', '-R', REPO, '-f', 'batch_size=36', '-f', 'start_fresh=false')
# ---- wave 3: Process tiles to parquets (only tiles with an ancillary commit; batch_size 72 -> ~60 jobs of ~1.2 h)
# gh('workflow', 'run', 'process_tiles.yml', '-R', REPO, '-f', 'batch_size=72', '-f', 'keep_pixels=false', '-f', 'start_fresh=false')

In [ ]:
# remaining work (the dispatchers' view) — re-run until both are 0, re-dispatching in between
done_anc, done_part = datacube.completed_ancillary_tiles(config), datacube.completed_partials_tiles(config)
wave2 = [t for t in tiles if t not in done_anc]
wave3 = [t for t in tiles if t in done_anc and t not in done_part]
print(f"wave 2 remaining {len(wave2)} | wave 3 remaining {len(wave3)} (blocked by wave 2: {len(wave2)}) | ancillary {len(done_anc)} | partials {len(done_part)}")
wave2[:10], wave3[:10]

## 4. One tile end to end

The same functions the workers call, one tile at a time, so every product can be looked at: the
ancillary window on the dataset grid (wave 2), the in-memory pixel table on the UTM grid and the tile's
partial sums (wave 3), and a single-tile reduce. Needs the Azure SAS token; building the ancillary window
needs Earth Engine only if the tile has no commit yet.

In [ ]:
row, col = 16, 152   # White Sea coast, Russia: a dry-run tile with a v9 counterpart (v9 tile (r, c) == v10 tile (r+2, c))
repo = datacube.open_ancillary_repo(config) if datacube.ancillary_repo_exists(config) else None
print('ancillary committed:', repo is not None and datacube.ancillary_tile_complete(config, row, col, repo),
      '| partials exist:', fs.exists(datacube.partials_tile_path(config, row, col)),
      '| pixel table exists:', fs.exists(datacube.parquet_tile_path(config, row, col)))

### Wave 2 — the ancillary window on the dataset grid

The eight source layers on the tile's 2048 × 2048 window of the dataset grid (EPSG:4326, 0.00072°):
Copernicus DEM, CHILI (native lattice, ÷255), snow class, WorldCover, forest cover and the GMBA /
BasinATLAS (level 6) / continent ids. Stored with compact integer encodings in the grid generation's
icechunk repository, one chunk and one commit per tile; `open_ancillary_window` decodes them back.

In [ ]:
if repo is not None and datacube.ancillary_tile_complete(config, row, col, repo):
    window = datacube.open_ancillary_window(config, row, col, repo)
else:
    print('EE as', settings.initialize_earthengine())                        # CHILI needs Earth Engine
    window = datacube.build_ancillary_window(config, row, col, log=print)   # ~2-3 min: DEM STAC + EE CHILI + vector rasterization
    # to commit it (the workers do this): repo = datacube.initialize_ancillary_store(config); datacube.write_ancillary_tile(config, repo, row, col, window)
print('basin layer:', settings.BASIN_ATLAS_LAYER)
window

In [ ]:
window[['dem', 'chili', 'snow_classification', 'forest_cover_fraction']].to_array().plot.imshow(col='variable', col_wrap=2, robust=True, figsize=(10, 9))

### Wave 3 — the pixel table on the UTM grid (in memory) and its partial sums

`tabulate_tile` reprojects the store window and the ancillary window onto the tile's 80 m UTM grid
(onset, DEM, CHILI, forest cover bilinear; categorical and id layers nearest), derives slope and aspect
from the UTM DEM and joins them (exact alignment); `aggregate.tile_partials` turns the table into
reducible sums per (filter, unit, bins, CHILI class). The worker writes only the partials (the pixel
table is opt-in: `keep_pixels`).

In [ ]:
global_ds = config.open_runoff_onset_dataset(chunks=None, mask_and_scale=True)
df = datacube.tabulate_tile(config, row, col, window, global_ds=global_ds)
print(f"{len(df):,} pixels x {len(df.columns)} columns on {df.attrs['utm_crs']} {df.attrs['utm_shape']}")
df.head()

In [ ]:
partials = aggregate.tile_partials(df, config.water_years)
print(f"{len(partials)} partial rows x {len(partials.columns)} columns")
partials.groupby(['filter_tag', 'unit_type']).size()

In [ ]:
# one partials row, read: this many pixels of this unit / bins / CHILI class, and their sums
partials.iloc[0]

In [ ]:
# what the reduce will see for this tile alone: the level-5 basin cube derived from the level-6 rows
tile_cube = aggregate.reduce_partials(partials, 'river_basins', 'fcf_lte_50', config.water_years)
aggregate.collapse(tile_cube)['runoff_onset_median'].plot(x='elevation', hue='river_basin', marker='o')

In [ ]:
# write the tile's products exactly as the wave-3 worker would (it skips tiles whose partials exist)
# datacube.process_tile(config, row, col, global_ds=global_ds, keep_pixels=False, log=print, repo=repo)

### Comparison with the v9 pixel table

The v9 per-tile parquets (`settings.V9_TILE_PARQUET_PREFIX`) are kept as the reference until the v10
campaign is validated: static ancillary columns must match, onset columns shift by the WY2025 signal
only. (v9 stored basins at HydroBASINS level 5, so `PFAF_ID // 10` here corresponds to the v9 id.)

In [ ]:
v9_df = pd.read_parquet(f"{settings.V9_TILE_PARQUET_PREFIX}/tile_{row-2:03d}_{col:03d}.parquet",
                        filesystem=config.azure_blob_fs)
common = [c for c in v9_df.columns if c in df.columns and c not in ('tile_row', '__index_level_0__', 'PFAF_ID')]
print(f"v9 rows {len(v9_df):,} | {config.version} rows {len(df):,}")
pd.DataFrame({'v9_median': v9_df[common].median(numeric_only=True), f'{config.version}_median': df[common].median(numeric_only=True)}).round(2)

## 5. Local stages, once the fleet reports 0 remaining

Each takes minutes and overwrites its outputs; `--mirror` copies the cubes to
`snowmelt_runoff_onset_analysis/aggregated/<version>/` so they exist off this machine. The default reduce writes the
three cubes the notebooks read; `--groups river_basins_l6` (after `era5_zonal.py --units river_basins_l6`)
and `--groups continents_aspect` add the finer cubes on demand. Pixi tasks: `pixi run era5-zonal`,
`pixi run reduce`, `pixi run metrics`.

In [ ]:
def run(script, *args):
    cmd = [sys.executable, str(paths.PIPELINE / 'scripts' / script), *args]
    print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)

# run('era5_zonal.py')                       # E3: per-range + per-basin ERA5 anomaly zonal means (~8 min)
# run('reduce_partials.py', '--mirror')     # the three cubes for both filter tags (minutes)
# run('range_metrics.py')                   # the per-range metrics table (seconds)

In [ ]:
# check: the cubes and what they contain
for group in list(aggregate.DEFAULT_GROUPS) + [g for g in aggregate.GROUPS if g not in aggregate.DEFAULT_GROUPS]:
    p = paths.aggregate(group, VERSION)
    if p.exists():
        ds = xr.open_dataset(p)
        print(f"{group:18s} {dict(ds.sizes)}  ERA5 merged: {'temperature_2m' in ds}  {p.stat().st_size/1e6:.1f} MB")
    else:
        print(f"{group:18s} not built{' (on demand)' if group not in aggregate.DEFAULT_GROUPS else ' yet'}")